# NeuralJam — Fine-tuning MelodyRNN con tus frases

**Antes de empezar:**
1. Corré `python tools/export_for_training.py` en tu PC
2. Subí el ZIP generado a Google Drive
3. En Colab: Entorno de ejecución → Cambiar tipo → GPU T4
4. Corré las celdas en orden

**Tiempo estimado:** 20–40 minutos con GPU T4 gratuita

## 1. Instalar Magenta

In [ ]:
!pip install magenta==2.1.4 --quiet
print('Magenta instalado.')

## 2. Montar Google Drive y descomprimir frases

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import glob, os

# Cambiá esto al path de tu ZIP en Drive
ZIP_PATH = '/content/drive/MyDrive/neuraljam_training_*.zip'

zips = sorted(glob.glob(ZIP_PATH))
if not zips:
    raise FileNotFoundError(f'No se encontró ZIP en {ZIP_PATH}\nSubí el archivo a Drive primero.')

zip_file = zips[-1]  # el más reciente
print(f'Usando: {zip_file}')

!mkdir -p /content/midi_input
!unzip -q "{zip_file}" -d /content/midi_input

midis = glob.glob('/content/midi_input/user_*.mid')
print(f'Frases del usuario encontradas: {len(midis)}')

## 3. Convertir MIDIs a NoteSequences

In [ ]:
!mkdir -p /content/notesequences

!convert_dir_to_note_sequences \
  --input_dir=/content/midi_input \
  --output_file=/content/notesequences/notesequences.tfrecord \
  --recursive

print('NoteSequences generadas.')

## 4. Crear dataset de entrenamiento

In [ ]:
!mkdir -p /content/sequence_examples

!melody_rnn_create_dataset \
  --config=attention_rnn \
  --input=/content/notesequences/notesequences.tfrecord \
  --output_dir=/content/sequence_examples \
  --eval_ratio=0.1

print('Dataset listo.')

## 5. Descargar checkpoint base (attention_rnn)

In [ ]:
!mkdir -p /content/logdir/attention_rnn/train

# Checkpoint base de Magenta
!wget -q http://download.magenta.tensorflow.org/models/attention_rnn.mag \
  -O /content/attention_rnn.mag

# Extraer el checkpoint del bundle
import magenta
from magenta.models.shared import sequence_generator_bundle
bundle = sequence_generator_bundle.read_bundle_file('/content/attention_rnn.mag')

# Guardar checkpoint para poder hacer fine-tuning
checkpoint_dir = '/content/logdir/attention_rnn/train'
with open(f'{checkpoint_dir}/bundle.mag', 'wb') as f:
    f.write(bundle.SerializeToString())

print('Checkpoint base listo.')

## 6. Fine-tuning

In [ ]:
# Ajustá TRAINING_STEPS según cuántas frases tenés:
#   30  frases → 300  steps
#   150 frases → 1000 steps
#   300 frases → 2000 steps

TRAINING_STEPS = 500  # cambiá esto

!melody_rnn_train \
  --config=attention_rnn \
  --run_dir=/content/logdir/attention_rnn \
  --sequence_example_file=/content/sequence_examples/training_melodies.tfrecord \
  --hparams="batch_size=64,rnn_layer_sizes=[128,128]" \
  --num_training_steps={TRAINING_STEPS}

print(f'Fine-tuning completado ({TRAINING_STEPS} steps).')

## 7. Exportar el nuevo bundle .mag

In [ ]:
from datetime import date
output_name = f'neuraljam_{date.today().isoformat()}.mag'

!melody_rnn_generate \
  --config=attention_rnn \
  --run_dir=/content/logdir/attention_rnn \
  --bundle_file=/content/{output_name} \
  --save_generator_bundle

print(f'Bundle guardado: {output_name}')

## 8. Guardar en Drive y descargar

In [ ]:
import shutil
drive_output = f'/content/drive/MyDrive/{output_name}'
shutil.copy(f'/content/{output_name}', drive_output)
print(f'Guardado en Drive: {drive_output}')
print()
print('Próximos pasos:')
print(f'  1. Bajá {output_name} de Drive a C:/AI-Duet-Local/models_data/')
print(f'  2. En config.py cambiá:')
print(f'       "model_path": MODELS_DIR / "{output_name}"')
print(f'  3. Reiniciá NeuralJam — MelodyRNN ya habló con vos.')